# Generate regulator representations from ChromBERT

**Note**: The remaining examples show Bash command-line usage only for extracting regulator embeddings.

`embed_region` subcommand: Generate regulator embeddings for input regulators across user-specified regions.

For the Python API, see [`examples/api/embed_regulator.ipynb`](../api/embed_regulator.ipynb).

For Singularity, see [`singularity_use.ipynb`](./singularity_use.ipynb) for `singularity exec` with chrombert-tools.

For more details, please refer to the [`embed_regulator`](https://chrombert-tools.readthedocs.io/en/latest/commands/embed_regulator.html) command documentation.

In [1]:
### options parameter
!chrombert-tools embed_regulator -h 

Usage: chrombert-tools embed_regulator [OPTIONS]

  Extract regulator embeddings on specified regions. Supports both general and
  cell-specific modes.

Options:
  --region FILE                   Region file.  [required]
  --regulator TEXT                Regulators of interest, e.g. EZH2 or
                                  EZH2;BRD4. Use ';' to separate multiple
                                  regulators.  [required]
  --cell-type-bw FILE             Cell type accessibility BigWig file. Used
                                  for cell-specific mode.
  --cell-type-peak FILE           Cell type accessibility Peak BED file. Used
                                  for cell-specific mode.
  --ft-ckpt FILE                  Fine-tuned checkpoint. If provided, use
                                  cell-specific model and skip fine-tuning.
  --odir DIRECTORY                Output directory.  [default: ./output]
  --oname TEXT                    Output name of the regulator embeddings.
        

### Generate regulator embeddings (pre-trained and general)

In [ ]:
%%bash
# --region: focus regions
# --regulator: focus regulators
# --odir: output directory
# --genome: genome
# --resolution: resolution
chrombert-tools embed_regulator \
--region '../data/CTCF_ENCFF664UGR_sample100.bed' \
--regulator "EZH2;BRD4;CTCF;FOXA3;myod1;myF5" \
--odir "./output_emb_regulator_1kb" \
--genome "hg38" \
--resolution "1kb"

Region summary - total: 100, overlapping with ChromBERT: 100 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 0
Note: All regulator names were converted to lowercase for matching.
Regulator count summary - requested: 6, matched in ChromBERT: 5, not found: 1, not found regulator: ['foxa3']
ChromBERT regulators: /mnt/Storage/home/chenqianqian/.cache/chrombert/data/config/hg38_6k_regulators_list.txt
Load pretrained ckpt /mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt successfully!
Your supervised_file does not contain the 'label' column. Please verify whether ground truth column ('label') is required. If it is not needed, you may disregard this message.
Your supervised_file does not contain the 'label' column. Please verify whether ground truth column ('label') is required. If it is not needed, you may disregard this message.


Computing regulator embeddings: 100%|██████████| 25/25 [00:05<00:00,  4.91it/s]



Finished!
Focus region summary - total: 100, overlapping with ChromBERT: 100, non-overlapping: 0
Overlapping regions BED file: ./output_emb_regulator_1kb/overlap_region.bed
Non-overlapping regions BED file: ./output_emb_regulator_1kb/no_overlap_region.bed
Mean regulator embeddings saved to: ./output_emb_regulator_1kb/mean_regulator_emb.pkl
Region-aware regulator embeddings saved to: ./output_emb_regulator_1kb/region_aware_regulator_emb.hdf5
Embedding type: general


In [ ]:
# regulator_emb_mean.pkl: one 768-dim vector per regulator (averaged across all regions)
import pickle
with open("./output_emb_regulator_1kb/mean_regulator_emb.pkl", "rb") as f:
    mean_regulator_emb_dict = pickle.load(f)

for key, value in mean_regulator_emb_dict.items():
    print(key, value.shape)


ezh2 (768,)
myod1 (768,)
brd4 (768,)
ctcf (768,)
myf5 (768,)


In [6]:
# Python dictionary mapping each matched regulator to its Region-aware regulator embeddings
# shape (N_regions, 768)
import h5py
with h5py.File("./output_emb_regulator_1kb/region_aware_regulator_emb.hdf5", "r") as f:
    print(f.keys())
    print(f['emb'].keys())
    for i in f['emb'].keys():
        print(i, f['emb'][i].shape)

<KeysViewHDF5 ['emb', 'region']>
<KeysViewHDF5 ['brd4', 'ctcf', 'ezh2', 'myf5', 'myod1']>
brd4 (100, 768)
ctcf (100, 768)
ezh2 (100, 768)
myf5 (100, 768)
myod1 (100, 768)


### Generate cell-type-specific embeddings using a fine-tuned checkpoint

We use a fine-tuned checkpoint as input.

The `embed_regulator` subcommand uses this checkpoint to generate context-specific embeddings.

In [3]:
## fine-tuned a cell-type-specific model
# '''
# --odir: output directory
# --acc_signal1: cell-type-specific accessibility signal
# --acc_peak1: cell-type-specific peak
# --genome: genome
# --resolution: resolution
# '''
# !chrombert-tools region_activity_regression \
# --odir "./output_cell_specific_emb_train" \
# --acc_signal1 "../data/myoblast_ENCFF149ERN_signal.bigwig" \
# --acc_peak1 "../data/myoblast_ENCFF647RNC_peak.bed" \
# --genome "hg38" \
# --resolution "1kb"


In [ ]:
import glob
ft_ckpt_dir = "./output_cell_specific_emb_train/train/**/*.ckpt"  # Use checkpoints from embed_region.ipynb if available; otherwise, run the code above first

ft_ckpt = glob.glob(ft_ckpt_dir, recursive=True)[0]
ft_ckpt


'./output_cell_specific_emb_train/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=0-step=37.ckpt'

In [ ]:
# --region: focus regions
# --regulator: focus regulators
# --odir: output directory
# --genome: genome
# --resolution: resolution
# --ft-ckpt: path to the fine-tuned checkpoint

!chrombert-tools embed_regulator \
--region '../data/CTCF_ENCFF664UGR_sample100.bed' \
--regulator "EZH2;BRD4;CTCF;FOXA3;myod1;myF5" \
--odir "./output_emb_regulator_1kb_load_ft" \
--genome "hg38" \
--resolution "1kb" \
--ft-ckpt {ft_ckpt}



Region summary - total: 100, overlapping with ChromBERT: 100 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 0
Note: All regulator names were converted to lowercase for matching.
Regulator count summary - requested: 6, matched in ChromBERT: 5, not found: 1, not found regulator: ['foxa3']
ChromBERT regulators: /mnt/Storage/home/chenqianqian/.cache/chrombert/data/config/hg38_6k_regulators_list.txt
Using provided fine-tuned checkpoint: ./output_cell_specific_emb_train/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=0-step=37.ckpt
Load pretrained ckpt /mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt successfully!
Loading checkpoint from ./output_cell_specific_emb_train/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=0-step=37.ckpt
Loading from pl module, remove prefix 'model.'
Loadin

In [10]:
# regulator_emb_mean.pkl: one 768-dim vector per regulator (averaged across all regions)
import pickle
with open("./output_emb_regulator_1kb_load_ft/mean_regulator_emb.pkl", "rb") as f:
    mean_regulator_emb_dict2 = pickle.load(f)

for key, value in mean_regulator_emb_dict2.items():
    print(key, value.shape)


myod1 (768,)
brd4 (768,)
ezh2 (768,)
myf5 (768,)
ctcf (768,)


In [16]:
mean_regulator_emb_dict["myod1"]

array([-2.00714844e+00, -9.10444336e-01, -6.23415527e-01, -2.63640625e+00,
       -3.17535553e-01, -9.51425781e-01,  5.77041626e-02, -3.99008789e-01,
        5.66394043e-02,  1.28871155e+00,  1.35552734e+00,  1.43785156e+00,
       -1.16653320e+00,  8.17010498e-01,  1.14992187e+00,  1.71201660e+00,
        6.81149902e-01, -7.94629364e-01,  6.32702942e-01,  1.00140625e+00,
        5.96638184e-01, -3.69458008e-01,  1.31123657e-01, -7.75520020e-01,
        2.88800049e-01,  1.74105469e+00,  4.14042969e-01,  1.72488281e+00,
       -8.69281006e-01, -1.28226074e+00, -1.62515625e+00, -1.21843750e+00,
       -3.18720093e-01,  1.53009766e+00, -7.61884766e-01,  7.34122696e-01,
       -1.57476562e+00, -1.67339478e-01, -3.62263184e-01, -1.70914063e+00,
        5.93414307e-01, -6.88667908e-01, -1.42664063e+00,  1.46355469e+00,
        7.41562500e-01,  2.43287201e-01,  3.57158813e-01,  6.16872559e-01,
        9.48164062e-01,  7.90859375e-01, -1.81777344e+00, -3.42831421e-01,
        4.01247559e-01,  

In [15]:
mean_regulator_emb_dict2["myod1"]

array([-2.05085937e+00, -6.61312561e-01, -7.87341309e-01, -2.56292969e+00,
       -2.15845642e-01, -7.26225586e-01,  1.12674332e-01, -2.44242554e-01,
        7.88000488e-02,  1.09402100e+00,  1.38145508e+00,  1.60011719e+00,
       -1.21123047e+00,  7.91054687e-01,  9.68378906e-01,  1.66437500e+00,
        4.65019531e-01, -7.14401245e-01,  5.66979179e-01,  1.17439453e+00,
        3.93154907e-01, -4.25457764e-01,  3.63861084e-02, -6.71879883e-01,
        1.86912842e-01,  2.00511719e+00,  2.07645264e-01,  1.96289062e+00,
       -1.07225830e+00, -1.07841812e+00, -1.36597656e+00, -1.37207031e+00,
       -2.77866821e-01,  1.77527344e+00, -4.60074158e-01,  6.37563477e-01,
       -1.61382812e+00, -3.39229126e-01, -2.87722168e-01, -1.97074219e+00,
        3.02783203e-01, -7.95349121e-01, -1.53894531e+00,  1.48253906e+00,
        5.90205078e-01,  2.16322327e-01,  3.10146484e-01,  1.08128906e+00,
        1.08240234e+00,  8.47650146e-01, -1.65046875e+00, -3.66970520e-01,
        4.82966309e-01,  

In [7]:
# Python dictionary mapping each matched regulator to its Region-aware regulator embeddings
# shape (N_regions, 768)
import h5py
with h5py.File("./output_emb_regulator_1kb_load_ft/region_aware_regulator_emb.hdf5", "r") as f:
    print(f.keys())
    print(f['emb'].keys())
    for i in f['emb'].keys():
        print(i, f['emb'][i].shape)

<KeysViewHDF5 ['emb', 'region']>
<KeysViewHDF5 ['brd4', 'ctcf', 'ezh2', 'myf5', 'myod1']>
brd4 (100, 768)
ctcf (100, 768)
ezh2 (100, 768)
myf5 (100, 768)
myod1 (100, 768)


### Generate cell-type-specific embeddings

We use cell-type-specific chromatin accessibility peak and signal files as input.

The `embed_regulator` subcommand uses these data to build a cell-type-specific model and generate cell-type-specific embeddings.

In [ ]:
# --region: focus regions
# --regulator: focus regulators
# --odir: output directory
# --genome: genome
# --resolution: resolution
# --cell-type-bw: Cell-type-specific bigwig file
# --cell-type-peak: Cell-type-specific peak file

!chrombert-tools embed_regulator \
--region '../data/CTCF_ENCFF664UGR_sample100.bed' \
--regulator "EZH2;BRD4;CTCF;FOXA3;myod1;myF5" \
--odir "./output_emb_regulator_1kb_load_ft" \
--genome "hg38" \
--resolution "1kb" \
--cell-type-bw ../data/myoblast_ENCFF149ERN_signal.bigwig \
--cell-type-peak ../data/myoblast_ENCFF647RNC_peak.bed